In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv('C:/Users/Playdata/Downloads/final_EDA_df.csv')

In [ ]:
df.columns

In [ ]:
import pandas as pd

# 1. 먼저 '거래날짜' 컬럼을 날짜 형식으로 변환 (가장 중요!)
df['거래날짜'] = pd.to_datetime(df['거래날짜'])

# 2. 날짜 형식인 상태에서 가장 최근 날짜(Maximum) 확인
max_date = df['거래날짜'].max()

# 3. 이제 날짜 객체이므로 1년 전 날짜 계산이 가능합니다.
start_date = max_date - pd.DateOffset(years=1)

# 4. 1년치 데이터만 필터링
filtered_df = df[df['거래날짜'] >= start_date].reset_index(drop=True)

print(f"최근 날짜: {max_date}")
print(f"시작 날짜 (1년 전): {start_date}")
# print(filtered_df)
filtered_df = filtered_df.drop(['API ratio'],axis=1)
filtered_df.info()

# filtered_df.to_csv('C:/Users/Playdata/Downloads/one_year_churn.csv')


In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

cat_cols = ['상품그룹', '멤버십상태', '연령대']
le_dict = {}

# 1. filtered_df가 안전하게 복사본인지 확인 (중요!)
filtered_df = filtered_df.copy()

for col in cat_cols:
    le = LabelEncoder()
    # 2. .loc를 사용하여 명시적으로 해당 컬럼의 모든 행을 변환된 값으로 대입
    # astype(str)은 결측치가 있을 때 에러를 방지하거나 일관성을 위해 유지하는 것이 좋습니다.
    filtered_df.loc[:, col] = le.fit_transform(filtered_df[col].astype(str))
    
    # 역변환을 위해 딕셔너리에 저장
    le_dict[col] = le

# 3. 결과 확인
print(filtered_df[cat_cols].head())
print(filtered_df[cat_cols].dtypes)
filtered_df

In [ ]:
mapping_result = {}

for col, le in le_dict.items():
    # 클래스(문자)와 인덱스(숫자)를 매칭하여 딕셔너리 생성
    mapping_result[col] = dict(zip(le.classes_, range(len(le.classes_))))

# 출력해서 확인
import pprint
pprint.pprint(mapping_result)

### smoke 적용

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
import numpy as np

# 1. 범주형 컬럼 리스트 (가격은 제외)
cat_cols = ['상품그룹', '멤버십상태', '연령대']

# ---------------------------
# 1️⃣ 신규 고객 모델 (LightGBM)
# ---------------------------
X_new = filtered_df[['상품그룹', '멤버십상태', '연령대', '가격','패션뉴스구독여부']].copy()
y_new = filtered_df['신규고객이탈']

# 범주형 컬럼만 타입 변경 (가격은 수치형 그대로 유지)
for col in cat_cols:
    X_new[col] = X_new[col].astype('category')

# 데이터 분할
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(
    X_new, y_new, test_size=0.2, stratify=y_new, random_state=42
)
smote = SMOTE(random_state=42)
X_resample, y_resample = smote.fit_resample(Xn_tr,yn_tr)

print(f'resample 후 샘플 비율 : {np.bincount(y_resample)}')
X_resample.shape, y_resample.shape

# 데이터 분할 (학습, 검증)
# X_tr, X_eval, y_tr, y_eval = train_test_split(X_resample, y_resample, random_state=0)

Xn_train, Xn_eval, yn_train, yn_eval = train_test_split(X_resample, y_resample,random_state=0) 
eval_set = [(Xn_train,yn_train),(Xn_eval,yn_eval)]


In [ ]:
import optuna
import lightgbm as lgb
from lightgbm import LGBMClassifier

# 1. 데이터 10% 샘플링 (이것만 해도 시간 90% 단축)
X_mini = Xn_train.sample(frac=0.1, random_state=42)
y_mini = yn_train.loc[X_mini.index]

def objective(trial):
    params = {
        'n_estimators': 500, # 속도를 위해 하향
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.2), # 학습 속도 상향
        'num_leaves': trial.suggest_int('num_leaves', 31, 128),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'objective': 'binary',
        'metric': 'auc',
        'data_sample_strategy': 'goss', # 대용량 속도 특화 옵션
        'verbose': -1
    }

    model = LGBMClassifier(**params)
    model.fit(X_mini, y_mini, eval_set=[(Xn_eval, yn_eval)], 
              callbacks=[lgb.early_stopping(stopping_rounds=30)]) # 빨리 멈추기

    return roc_auc_score(yn_eval, model.predict_proba(Xn_eval)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10) # 10번만 딱 돌리기!



print(f"최고 AUC: {study.best_value}")
print(f"최적 파라미터: {study.best_params}")

In [ ]:
print(f'optuna 최적 파라미터 : {study.best_params}')

In [ ]:
# # LightGBM 모델 적용

lgbm = LGBMClassifier(
    learning_rate = 0.06,
    num_leaves = 109,
    eval_metric='binary_logloss',
    max_depth = 8,
    verbose = 1 # 경고 출력
)
lgbm.fit(Xn_train, yn_train, eval_set=[(Xn_eval, yn_eval)])
# eval_set = [(Xn_train,yn_train),(Xn_eval,yn_eval)]


In [ ]:
# 모델, 학습 점수 
lgbm.score(Xn_train,yn_train), lgbm.score(Xn_te, yn_te)


In [ ]:
# 신규 혼동 행렬, 정밀도, 재현율
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score
y_pred = lgbm.predict(Xn_te)
print('혼동행렬(Confusion Matrix) \n',confusion_matrix(y_pred=y_pred, y_true=yn_te),'\n')
print('\n',f'정밀도 : {precision_score(yn_te,y_pred)}')
print('\n',f'재현율 : {recall_score(yn_te,y_pred)}')
print('\n',f'정확도 : {accuracy_score(yn_te,y_pred)}')



In [ ]:
# classfication_report 
from sklearn.metrics import classification_report

print(classification_report(yn_te,y_pred))
preds_total = lgbm.predict_proba(Xn_te)[:, 1]
print(f"전체 고객 모델 AUC: {roc_auc_score(yn_te, preds_total):.4f}")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
import numpy as np

# 1. 범주형 컬럼 리스트 (가격은 제외)
cat_cols = ['상품그룹', '멤버십상태', '연령대']

# ---------------------------
# 1️⃣ 신규 고객 모델 (LightGBM)
# ---------------------------
X_total = filtered_df[['상품그룹', '멤버십상태', '연령대', '가격','패션뉴스구독여부']].copy()
y_total = filtered_df['전체고객이탈여부']

# 범주형 컬럼만 타입 변경 (가격은 수치형 그대로 유지)
for col in cat_cols:
    X_total[col] = X_total[col].astype('category')

# 데이터 분할
Xt_tr, Xt_te, yt_tr, yt_te = train_test_split(
    X_total, y_total, test_size=0.2, stratify=y_total, random_state=42
)
smote = SMOTE(random_state=42)
X_resample, y_resample = smote.fit_resample(Xt_tr,yt_tr)

print(f'resample 후 샘플 비율 : {np.bincount(y_resample)}')
X_resample.shape, y_resample.shape

# 데이터 분할 (학습, 검증)
# X_tr, X_eval, y_tr, y_eval = train_test_split(X_resample, y_resample, random_state=0)

Xt_train, Xt_eval, yt_train, yt_eval = train_test_split(X_resample, y_resample,random_state=0) 
eval_set = [(Xt_train,yt_train),(Xt_eval,yt_eval)]


In [ ]:
import optuna
import lightgbm as lgb
from lightgbm import LGBMClassifier

# 1. 데이터 10% 샘플링 (이것만 해도 시간 90% 단축)
Xt_mini = Xt_train.sample(frac=0.1, random_state=42)
yt_mini = yt_train.loc[Xt_mini.index]

def objective(trial):
    params = {
        'n_estimators': 500, # 속도를 위해 하향
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.2), # 학습 속도 상향
        'num_leaves': trial.suggest_int('num_leaves', 31, 128),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'objective': 'binary',
        'metric': 'auc',
        'data_sample_strategy': 'goss', # 대용량 속도 특화 옵션
        'verbose': -1
    }

    model = LGBMClassifier(**params)
    model.fit(Xt_mini, yt_mini, eval_set=[(Xt_eval, yt_eval)], 
              callbacks=[lgb.early_stopping(stopping_rounds=30)]) # 빨리 멈추기

    return roc_auc_score(yt_eval, model.predict_proba(Xt_eval)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10) # 10번만 딱 돌리기!



print(f"최고 AUC: {study.best_value}")
print(f"최적 파라미터: {study.best_params}")

In [ ]:
print(f'optuna 최적 파라미터 : {study.best_params}')

In [ ]:
# # LightGBM 모델 적용

lgbm_total = LGBMClassifier(
    learning_rate = 0.06,
    num_leaves = 109,
    eval_metric='binary_logloss',
    max_depth = 8,
    verbose = 1 # 경고 출력
)
lgbm_total.fit(Xt_train, yt_train, eval_set=[(Xt_eval, yt_eval)])
# eval_set = [(Xn_train,yn_train),(Xn_eval,yn_eval)]


In [ ]:
# 모델, 학습 점수 
lgbm_total.score(Xt_train,yt_train), lgbm_total.score(Xt_te, yt_te)


In [ ]:
# 신규 혼동 행렬, 정밀도, 재현율
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score
y_pred = lgbm_total.predict(Xt_te)
print('혼동행렬(Confusion Matrix) \n',confusion_matrix(y_pred=y_pred, y_true=yt_te),'\n')
print('\n',f'정밀도 : {precision_score(yt_te,y_pred)}')
print('\n',f'재현율 : {recall_score(yt_te,y_pred)}')
print('\n',f'정확도 : {accuracy_score(yt_te,y_pred)}')



In [ ]:
# classfication_report 
from sklearn.metrics import classification_report

print(classification_report(yt_te,y_pred))
preds_total = lgbm.predict_proba(Xt_te)[:, 1]
print(f"전체 고객 모델 AUC: {roc_auc_score(yt_te, preds_total):.4f}")

In [ ]:
from sklearn.preprocessing import Binarizer
from sklearn.metrics import accuracy_score

def evaluate_binary_clf(yt_te,y_pred):
    print('혼동행렬')
    print(confusion_matrix(yt_te,y_pred),'\n')
    print(f'정확도 : {accuracy_score(yt_te,y_pred):.4f}/정밀도{precision_score(yt_te,y_pred)}/재현율{recall_score(yt_te,y_pred)}')

pred_proba = model_total.predict_proba(Xt_te)
    

In [ ]:

from sklearn.preprocessing import Binarizer
from sklearn.metrics import accuracy_score
pred_proba = model_total.predict_proba(Xt_te)
pred_proba_1 = pred_proba[:,1].reshape(-1,1)

thresholds = [0.0, 0.1,0.3,0.5,0.6,0.8]

for threshold in thresholds:
    binarizer = Binarizer(threshold=threshold)
    custom_pred = binarizer.fit_transform(pred_proba_1)
    evaluate_binary_clf(yt_te, custom_pred)

from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

precisions, recalls, thresholds = precision_recall_curve(yt_te, pred_proba_1) # 정밀도, 재현율은 마지막에 극단적으로 낮은 임계치 값을 적용한 값을 보여줌 ([:-1]) 1개 뺴주기

plt.figure(figsize=(6,4))
plt.plot(thresholds, precisions[:-1], linestyle='--', label='Precision')
plt.plot(thresholds, recalls[:-1], label = 'Recall')
plt.xlabel('Threshold')
plt.ylabel('precision and Recall value')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score


pred_proba = model_total.predict_proba(Xt_te)
pred_proba_1 = pred_proba[:,1].reshape(-1,1)


# fpr, tpr, thresholds = roc_curve(y_test, y_pred)

from sklearn.metrics import auc
fpr, tpr, thresholds = roc_curve(yt_te, pred_proba_1)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))

# ROC 곡선 그리기 (굵기 조절 및 색상 강조)
plt.plot(fpr, tpr, color='darkorange', lw=3, label=f'ROC curve (AUC = {roc_auc:.2f})')

# 대각선 점선 (Random Guess 기준선)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')


plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('LightGBM ROC Curve', fontsize=15)
plt.legend(loc="lower right", fontsize=12)

# plt.savefig('./graphs/RF_balanced_ROC_curve.png', dpi=300, bbox_inches='tight')

plt.show()

# roc_auc_score(y_test, pred_proba[:, 1])
roc_auc_score(yt_te, pred_proba_1)


In [ ]:
# ==============================
# Feature Importance
# ==============================
importances = model_total.feature_importances_

feature_importance_df = pd.DataFrame({
    'feature': X_new.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feature_importance_df.head(10))

import seaborn as sns
# 한글 설정 (윈도우)
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 깨짐 방지
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 기본 한글 폰트

plt.figure(figsize=(8,6))
sns.barplot(
    data=feature_importance_df.head(15),
    x='importance',
    y='feature'
)
plt.title('Feature Importances (LightGBM)')
plt.xlabel('Importance',fontsize=12)
plt.ylabel('Feature',fontsize=12)
plt.show()